# Feedforward Neural Networks
Accompanies the notes on **Feedforward Neural Networks**. Everything uses NumPy and scikit-learn.

## 1. Backpropagation by hand, checked in NumPy

In [ ]:
import numpy as np

sigmoid = lambda z: 1 / (1 + np.exp(-z))
x = np.array([0.5, 0.8]); y = 1.0
W1 = np.array([[0.2, 0.4], [-0.3, 0.5]]); b1 = np.array([0.1, -0.2])
W2 = np.array([[0.6, -0.4], [0.3, 0.7]]); b2 = np.array([0.2, 0.1])
W3 = np.array([[0.5, -0.6]]);             b3 = np.array([0.3])

def forward(W1, b1, W2, b2, W3, b3):
    a1 = sigmoid(W1 @ x + b1)
    a2 = sigmoid(W2 @ a1 + b2)
    a3 = sigmoid(W3 @ a2 + b3)
    return a1, a2, a3

a1, a2, y_hat = forward(W1, b1, W2, b2, W3, b3)
loss = 0.5 * (y_hat[0] - y) ** 2
print("a1:", a1.round(4), " a2:", a2.round(4), " y_hat:", y_hat.round(4), " loss:", round(loss, 6))

# backward pass: the four equations
d3 = (y_hat - y) * y_hat * (1 - y_hat)
d2 = (W3.T @ d3) * a2 * (1 - a2)
d1 = (W2.T @ d2) * a1 * (1 - a1)
grads = {"W3": np.outer(d3, a2), "b3": d3, "W2": np.outer(d2, a1), "b2": d2, "W1": np.outer(d1, x), "b1": d1}
print("delta3:", d3.round(6), " delta2:", d2.round(6), " delta1:", d1.round(6))
for name in ["W3", "W2", "W1"]:
    print(f"dL/d{name}:\n", grads[name].round(6))

# check one gradient with a finite difference
eps = 1e-6
W1p = W1.copy(); W1p[1, 1] += eps
numeric = (0.5 * (forward(W1p, b1, W2, b2, W3, b3)[2][0] - y) ** 2 - loss) / eps
print(f"dL/dW1[2,2]: backprop {grads['W1'][1, 1]:.6f}, finite difference {numeric:.6f}")

# one gradient descent step with eta = 0.5
eta = 0.5
new = [W1 - eta * grads["W1"], b1 - eta * grads["b1"], W2 - eta * grads["W2"], b2 - eta * grads["b2"],
       W3 - eta * grads["W3"], b3 - eta * grads["b3"]]
y_new = forward(*new)[2]
print("after one step: y_hat", y_new.round(4), " loss", round(0.5 * (y_new[0] - y) ** 2, 6))

## 2. XOR: logistic regression vs a hand-made network vs a trained MLP

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]); y_xor = np.array([0, 1, 1, 0])
print("logistic regression predictions:", LogisticRegression().fit(X_xor, y_xor).predict(X_xor))

relu = lambda z: np.maximum(0, z)
H = relu(X_xor @ np.array([[1, 1], [1, 1]]).T + np.array([0, -1]))   # h1 = relu(x1 + x2), h2 = relu(x1 + x2 - 1)
out = H @ np.array([1, -2])                                          # y = h1 - 2 h2
print("hidden layer:\n", H, "\nhand-made network output:", out)

mlp = MLPClassifier(hidden_layer_sizes=(4,), activation="tanh", solver="lbfgs", random_state=1).fit(X_xor, y_xor)
print("MLP predictions:", mlp.predict(X_xor))

## 3. Activation functions and their derivatives

In [ ]:
import matplotlib.pyplot as plt

z = np.linspace(-4, 4, 400)
s = sigmoid(z)
acts = {"sigmoid": (s, s * (1 - s)), "tanh": (np.tanh(z), 1 - np.tanh(z) ** 2),
        "ReLU": (relu(z), (z > 0).astype(float)), "Swish/SiLU": (z * s, s + z * s * (1 - s))}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, (f, df) in acts.items():
    axes[0].plot(z, f, label=name); axes[1].plot(z, df, label=name)
axes[0].set_title("activation f(z)"); axes[1].set_title("derivative f'(z)")
for ax in axes: ax.axhline(0, color="gray", lw=0.5); ax.legend(); ax.set_xlabel("z")
plt.tight_layout(); plt.show()

## 4. Initialisation: variance through 10 ReLU layers

In [ ]:
rng = np.random.default_rng(0)
width, depth = 512, 10
X0 = rng.normal(size=(1000, width))
for name, std in [("N(0, 1)", 1.0), ("Xavier, var 1/n", np.sqrt(1 / width)), ("He, var 2/n", np.sqrt(2 / width))]:
    a = X0; variances = []
    for layer in range(depth):
        a = relu(a @ rng.normal(0, std, size=(width, width)))
        variances.append(a.var())
    print(f"{name:16s} variance after layers 1, 5, 10: {variances[0]:.3g}, {variances[4]:.3g}, {variances[9]:.3g}")

## 5. MLP on the bank data: how to prepare the inputs

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

bd = pd.read_csv("../data/bd_train.csv").drop(columns=["REF_NO", "post_code", "post_area"])
y_b = (bd.pop("Revenue.Grid") == 1).astype(int)
numeric = bd.select_dtypes("number").columns.tolist()
X_b = pd.get_dummies(bd, drop_first=True, dtype=float)
x_train, x_test, y_train, y_test = train_test_split(X_b, y_b, test_size=0.25, random_state=1, stratify=y_b)

preprocessing = {"no scaling": None,
                 "scale every column": StandardScaler(),
                 "scale numeric, dummies 0/1": ColumnTransformer([("num", StandardScaler(), numeric)], remainder="passthrough")}

def fit_mlp(pre, alpha):
    mlp = MLPClassifier(hidden_layer_sizes=(64, 32), alpha=alpha, early_stopping=True, validation_fraction=0.15,
                        n_iter_no_change=20, max_iter=1000, random_state=0)
    model = make_pipeline(pre, mlp) if pre is not None else mlp
    model.fit(x_train, y_train)
    return model, roc_auc_score(y_test, model.predict_proba(x_test)[:, 1])

for alpha in [0.0001, 1.0]:
    for name, pre in preprocessing.items():
        _, auc = fit_mlp(pre, alpha)
        print(f"alpha={alpha:<6} {name:28s}: test AUC {auc:.4f}")

## 6. Training curve with early stopping

In [ ]:
model, _ = fit_mlp(preprocessing["scale numeric, dummies 0/1"], alpha=1.0)
mlp = model[-1]
fig, ax1 = plt.subplots(figsize=(7, 3.5))
ax1.plot(mlp.loss_curve_, label="training loss"); ax1.set_xlabel("epoch"); ax1.set_ylabel("training loss")
ax2 = ax1.twinx(); ax2.plot(mlp.validation_scores_, color="crimson", label="validation accuracy"); ax2.set_ylabel("validation accuracy")
fig.legend(loc="center right"); plt.tight_layout(); plt.show()
print("epochs run:", mlp.n_iter_, " best validation accuracy:", round(mlp.best_validation_score_, 4),
      "at epoch", int(np.argmax(mlp.validation_scores_)) + 1)

## 7. MLP regression on the loan data

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error

ld = pd.read_csv("../data/loan_data_train.csv")
ld["Interest.Rate"] = pd.to_numeric(ld["Interest.Rate"].str.replace("%", ""), errors="coerce")
ld["FICO"] = ld["FICO.Range"].str.split("-", expand=True).astype(float).mean(axis=1)
ld["Loan.Length"] = pd.to_numeric(ld["Loan.Length"].str.replace(" months", ""), errors="coerce")
cols = ["FICO", "Loan.Length", "Amount.Requested", "Inquiries.in.the.Last.6.Months",
        "Monthly.Income", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]
for col in cols:
    ld[col] = pd.to_numeric(ld[col], errors="coerce")
ld = ld[cols + ["Interest.Rate"]].dropna()
xr_train, xr_test, yr_train, yr_test = train_test_split(ld[cols], ld["Interest.Rate"], test_size=0.2, random_state=2)
for hidden in [(16,), (64, 32)]:
    reg = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=hidden, alpha=1.0, early_stopping=True,
                                                       max_iter=2000, n_iter_no_change=30, random_state=0))
    reg.fit(xr_train, yr_train)
    print(f"hidden layers {str(hidden):9s}: test MAE {mean_absolute_error(yr_test, reg.predict(xr_test)):.3f}")

## Try it yourself
1. Target y = 0 in section 1.
2. `hidden_layer_sizes=(128, 64, 32)` in section 5.